# Preparo

## Bibliotecas

In [1]:
import os
import sys

import pandas as pd
import plotly.express as px

In [2]:
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

In [3]:
from utils.add_country_name import add_country_name
from utils.adjust_cuisines import adjust_cuisines
from utils.clean_data import clean_data
from utils.convert_to_usd import convert_to_usd
from utils.create_color_name import create_color_name
from utils.create_price_tye import create_price_tye
from utils.create_unique_restaurant_name import create_unique_restaurant_name
from utils.rename_columns import rename_columns

## Baixando o Dataset

In [4]:
file_path = os.path.join("..", "database", "zomato.csv")

print(f"Tentando ler o arquivo em: {os.path.abspath(file_path)}")

try:
    df = pd.read_csv(file_path, 
                    encoding="utf-8", 
                    on_bad_lines="skip")
    print("Sucesso! O DataFrame foi carregado.")
except Exception as e:
    print(f"Erro: {e}")

Tentando ler o arquivo em: c:\Users\Admin\Documents\Comunidade DS\Analise de Dados Com Python\Empresa Fome Zero\database\zomato.csv
Sucesso! O DataFrame foi carregado.


## Limpando o Dataset

In [5]:
df = (df.pipe(rename_columns)
                .pipe(add_country_name)
                .pipe(clean_data)
                .pipe(convert_to_usd)
                .pipe(create_price_tye)
                .pipe(create_color_name)
                .pipe(adjust_cuisines)
                .pipe(create_unique_restaurant_name))

# Geral

In [6]:
# 1. Realizando os cálculos
qtd_restaurantes = len(df)
qtd_paises = df["country"].nunique()
qtd_cidades = df["city"].nunique()
soma_avaliacoes = df["votes"].sum()
qtd_culinarias = df["cuisines"].nunique()

# 2. Formatando as avaliações para o padrão brasileiro (ex: 4.194.530)
soma_avaliacoes_formatada = f"{soma_avaliacoes:,.0f}".replace(",", ".")

# 3. Exibindo os resultados na tela (print)
print("O Melhor lugar para encontrar seu mais novo restaurante favorito!")
print("Temos as seguintes marcas dentro da nossa plataforma:\n")

print(f"Restaurantes Cadastrados: {qtd_restaurantes}")
print(f"Países Cadastrados: {qtd_paises}")
print(f"Cidades Cadastradas: {qtd_cidades}")
print(f"Avaliações Feitas na Plataforma: {soma_avaliacoes_formatada}")
print(f"Tipos de Culinárias Oferecidas: {qtd_culinarias}")

O Melhor lugar para encontrar seu mais novo restaurante favorito!
Temos as seguintes marcas dentro da nossa plataforma:

Restaurantes Cadastrados: 6942
Países Cadastrados: 15
Cidades Cadastradas: 125
Avaliações Feitas na Plataforma: 4.195.634
Tipos de Culinárias Oferecidas: 165


In [10]:
import folium
from folium.plugins import MarkerCluster

# 1. Criando o mapa base (centralizado no "meio" do mundo com zoom afastado)
mapa = folium.Map(location=[20, 0], zoom_start=2)

# 2. Criando a camada de Agrupamento (Cluster)
cluster = MarkerCluster().add_to(mapa)

# 3. Adicionando cada restaurante dentro do cluster
# O iterrows vai passar linha por linha no seu DataFrame
for index, linha in df.iterrows():
    folium.Marker(
        location=[linha["latitude"], linha["longitude"]],
        popup=linha["restaurant_name"], # Mostra o nome do restaurante ao clicar no pino final
        tooltip=linha["cuisines"]      # Mostra a culinária ao passar o mouse
    ).add_to(cluster)

# 4. Exibindo o mapa no Notebook (é só chamar a variável)
mapa.save("meu_mapa_interativo.html")